# 🛡️ ICS Network Anomaly Detection
## Notebook 03: SHAP Explainability & Model Interpretability

**Objective**: Explain model predictions using SHAP (SHapley Additive exPlanations) for audit trails and compliance

**Why SHAP for ICS Security?**
- ✅ **Audit Requirements**: Explain why an attack was detected
- ✅ **IEC 62443 Compliance**: Document decision logic
- ✅ **SOC Teams**: Help analysts understand alerts
- ✅ **Model Trust**: Build confidence in predictions

**Applications**: Schneider Electric & Yokogawa Security Operations Centers

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
import joblib
import shap

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully!")
print(f"   SHAP version: {shap.__version__}")

## 1️⃣ Load Trained Models

In [ ]:
# Load XGBoost model (primary model)
model_path = Path('../models/xgboost_ics_detector.pkl')
scaler_path = Path('../models/feature_scaler.pkl')
features_path = Path('../models/feature_names.txt')

if not model_path.exists():
    print("❌ Model not found. Please run notebook 02 first.")
    raise FileNotFoundError("Run 02_ics_model_training.ipynb first")

# Load model and scaler
xgb_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# Load feature names
with open(features_path, 'r') as f:
    feature_names = [line.strip() for line in f]

print("✅ Models loaded:")
print(f"   • XGBoost model with {len(feature_names)} features")
print(f"   • StandardScaler for feature preprocessing")

## 2️⃣ Load Test Data

In [ ]:
# Load features and labels
features = pd.read_csv('../data/processed/ics_features.csv')
labels = pd.read_csv('../data/processed/ics_labels.csv')['label']

print(f"✅ Loaded {len(features):,} samples")

# Scale features
features_scaled = pd.DataFrame(
    scaler.transform(features),
    columns=features.columns,
    index=features.index
)

print(f"✅ Features scaled")

## 3️⃣ Create SHAP Explainer

SHAP uses game theory to explain predictions

In [ ]:
# Initialize SHAP explainer
print("🔍 Creating SHAP explainer...")
print("   (This may take 1-2 minutes)\n")

# Use a sample for faster computation
sample_size = 500
sample_indices = np.random.choice(len(features_scaled), sample_size, replace=False)
X_sample = features_scaled.iloc[sample_indices]
y_sample = labels.iloc[sample_indices]

# Create TreeExplainer (optimized for tree-based models)
explainer = shap.TreeExplainer(xgb_model)

print("✅ SHAP explainer created")
print(f"   Using {sample_size} samples for analysis")

## 4️⃣ Calculate SHAP Values

In [ ]:
# Calculate SHAP values
print("📊 Calculating SHAP values...")
shap_values = explainer.shap_values(X_sample)

print(f"✅ SHAP values calculated")
print(f"   Shape: {shap_values.shape}")
print(f"   Features: {shap_values.shape[1]}")
print(f"   Samples: {shap_values.shape[0]}")

## 5️⃣ Global Feature Importance

Which features are most important overall?

In [ ]:
# Summary plot - shows feature importance
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False, max_display=15)
plt.title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../results/shap_global_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/shap_global_importance.png")

## 6️⃣ Feature Impact Distribution

How do feature values affect predictions?

In [ ]:
# Summary plot with feature values
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
plt.title('Feature Impact on Attack Detection', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../results/shap_feature_impact.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/shap_feature_impact.png")
print("\n💡 Interpretation:")
print("   • Red points: High feature values")
print("   • Blue points: Low feature values")
print("   • Position on X-axis: Impact on prediction")
print("   • Right = pushes toward 'Attack', Left = pushes toward 'Normal'")

## 7️⃣ Explain Individual Predictions

Deep dive into specific attack detections

In [ ]:
# Find interesting examples
predictions = xgb_model.predict(X_sample)

# Find a correctly detected attack
attack_indices = np.where((y_sample == 1) & (predictions == 1))[0]
if len(attack_indices) > 0:
    attack_idx = attack_indices[0]
    print(f"📍 Analyzing Attack Detection (sample {attack_idx})")
    print(f"   Actual: Attack | Predicted: Attack ✅")
else:
    attack_idx = 0
    print("⚠️  No correctly detected attacks in sample")

# Find a normal traffic sample
normal_indices = np.where((y_sample == 0) & (predictions == 0))[0]
if len(normal_indices) > 0:
    normal_idx = normal_indices[0]
    print(f"\n📍 Analyzing Normal Traffic (sample {normal_idx})")
    print(f"   Actual: Normal | Predicted: Normal ✅")
else:
    normal_idx = 1
    print("⚠️  No correctly detected normal traffic in sample")

In [ ]:
# Waterfall plot for attack detection
if len(attack_indices) > 0:
    print("\n🎯 Attack Detection Explanation:")
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[attack_idx],
            base_values=explainer.expected_value,
            data=X_sample.iloc[attack_idx],
            feature_names=X_sample.columns.tolist()
        ),
        max_display=15,
        show=False
    )
    plt.title('Why This Traffic Was Classified as Attack', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/shap_attack_explanation.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: results/shap_attack_explanation.png")
    
    print("\n💡 Reading the Waterfall Plot:")
    print("   • Starts at E[f(X)] (base prediction for all data)")
    print("   • Each bar shows one feature's contribution")
    print("   • Red bars push toward 'Attack' (class 1)")
    print("   • Blue bars push toward 'Normal' (class 0)")
    print("   • Final value = f(X) (actual prediction)")

In [ ]:
# Waterfall plot for normal traffic
if len(normal_indices) > 0:
    print("\n✅ Normal Traffic Explanation:")
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[normal_idx],
            base_values=explainer.expected_value,
            data=X_sample.iloc[normal_idx],
            feature_names=X_sample.columns.tolist()
        ),
        max_display=15,
        show=False
    )
    plt.title('Why This Traffic Was Classified as Normal', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/shap_normal_explanation.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: results/shap_normal_explanation.png")

## 8️⃣ Force Plot - Interactive Explanation

In [ ]:
# Force plot for attack sample
if len(attack_indices) > 0:
    print("🎨 Force Plot - Attack Detection:")
    shap.force_plot(
        explainer.expected_value,
        shap_values[attack_idx],
        X_sample.iloc[attack_idx],
        feature_names=X_sample.columns.tolist(),
        matplotlib=True,
        show=False
    )
    plt.tight_layout()
    plt.savefig('../results/shap_force_attack.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: results/shap_force_attack.png")
    
    print("\n💡 Force Plot Interpretation:")
    print("   • Red features: Push prediction toward 'Attack'")
    print("   • Blue features: Push prediction toward 'Normal'")
    print("   • Width of bar: Magnitude of impact")
    print("   • Base value → Output value shows net effect")

## 9️⃣ Dependence Plots - Feature Interactions

In [ ]:
# Get top 3 most important features
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_features_idx = np.argsort(mean_abs_shap)[-3:][::-1]
top_feature_names = [X_sample.columns[i] for i in top_features_idx]

print(f"🔍 Top 3 Most Important Features:")
for i, name in enumerate(top_feature_names, 1):
    print(f"   {i}. {name}")

# Create dependence plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature_idx in enumerate(top_features_idx):
    feature_name = X_sample.columns[feature_idx]
    
    # Plot on subplot
    plt.sca(axes[idx])
    shap.dependence_plot(
        feature_idx,
        shap_values,
        X_sample,
        feature_names=X_sample.columns.tolist(),
        show=False,
        ax=axes[idx]
    )
    axes[idx].set_title(f'Impact of {feature_name}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/shap_dependence_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/shap_dependence_plots.png")

print("\n💡 Dependence Plot Interpretation:")
print("   • X-axis: Feature value")
print("   • Y-axis: SHAP value (impact on prediction)")
print("   • Color: Another feature's value (interaction effect)")
print("   • Shows how feature value affects prediction")

## 🔟 Decision Plot - Multiple Samples

In [ ]:
# Decision plot for first 20 samples
n_samples = min(20, len(X_sample))

plt.figure(figsize=(12, 8))
shap.decision_plot(
    explainer.expected_value,
    shap_values[:n_samples],
    X_sample.iloc[:n_samples],
    feature_names=X_sample.columns.tolist(),
    show=False,
    feature_display_range=slice(-1, -16, -1)  # Top 15 features
)
plt.title(f'Decision Paths for {n_samples} Network Flows', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/shap_decision_plot.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Saved: results/shap_decision_plot.png")

print("\n💡 Decision Plot Interpretation:")
print("   • Each line represents one prediction")
print("   • Starts at expected value (bottom)")
print("   • Moves up/down based on feature values")
print("   • Final position = prediction score")
print("   • Red lines = Attack predictions")
print("   • Blue lines = Normal predictions")

## 1️⃣1️⃣ Generate Audit Report

In [ ]:
# Create audit report for detected attacks
if len(attack_indices) > 0:
    print("📋 Generating Audit Report for Attack Detections...\n")
    
    # Analyze first 5 detected attacks
    num_reports = min(5, len(attack_indices))
    
    audit_reports = []
    
    for i in range(num_reports):
        idx = attack_indices[i]
        
        # Get SHAP values for this sample
        sample_shap = shap_values[idx]
        sample_features = X_sample.iloc[idx]
        
        # Get top contributing features
        feature_contributions = pd.DataFrame({
            'Feature': X_sample.columns,
            'Value': sample_features.values,
            'SHAP_Value': sample_shap
        }).sort_values('SHAP_Value', key=abs, ascending=False).head(10)
        
        report = {
            'sample_id': idx,
            'prediction': 'Attack',
            'confidence': float(xgb_model.predict_proba(X_sample.iloc[[idx]])[0][1]),
            'top_features': feature_contributions.to_dict('records')
        }
        
        audit_reports.append(report)
        
        print(f"Attack #{i+1} (Sample {idx}):")
        print(f"   Confidence: {report['confidence']:.1%}")
        print(f"   Top 3 Contributing Features:")
        for j, row in feature_contributions.head(3).iterrows():
            print(f"      • {row['Feature']}: {row['Value']:.3f} (SHAP: {row['SHAP_Value']:.3f})")
        print()
    
    # Save audit report
    import json
    with open('../results/shap_audit_report.json', 'w') as f:
        json.dump(audit_reports, f, indent=2)
    
    print(f"✅ Saved: results/shap_audit_report.json")
    print(f"   Contains explanations for {num_reports} attack detections")

## 1️⃣2️⃣ Key Insights Summary

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS FROM SHAP ANALYSIS")
print("="*80)

# Top features globally
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': X_sample.columns,
    'Mean_SHAP': mean_abs_shap
}).sort_values('Mean_SHAP', ascending=False)

print("\n🔝 Top 10 Most Important Features for Attack Detection:")
for i, row in feature_importance.head(10).iterrows():
    print(f"   {i+1:2d}. {row['Feature']:30s} (SHAP: {row['Mean_SHAP']:.4f})")

# Attack patterns
attack_shap = shap_values[y_sample == 1]
normal_shap = shap_values[y_sample == 0]

print("\n🎯 Attack vs Normal Patterns:")
print(f"   • Attack samples analyzed: {len(attack_shap)}")
print(f"   • Normal samples analyzed: {len(normal_shap)}")
print(f"   • Avg SHAP magnitude (Attack): {np.abs(attack_shap).mean():.4f}")
print(f"   • Avg SHAP magnitude (Normal): {np.abs(normal_shap).mean():.4f}")

print("\n🏭 Industrial Applications:")
print("   ✅ Audit trail for security incidents")
print("   ✅ Explain alerts to SOC analysts")
print("   ✅ IEC 62443 compliance documentation")
print("   ✅ Fine-tune detection rules based on feature importance")
print("   ✅ Train security teams on attack patterns")

print("\n💾 Generated Reports:")
print("   • shap_global_importance.png - Overall feature importance")
print("   • shap_feature_impact.png - Feature value effects")
print("   • shap_attack_explanation.png - Attack detection reasoning")
print("   • shap_normal_explanation.png - Normal traffic reasoning")
print("   • shap_dependence_plots.png - Feature interactions")
print("   • shap_decision_plot.png - Decision paths")
print("   • shap_audit_report.json - Detailed audit logs")

print("\n" + "="*80)
print("✅ SHAP analysis completed!")
print("➡️  Use these explanations for:")
print("   • Security incident reports")
print("   • Compliance audits")
print("   • Model improvement")
print("   • SOC team training")
print("="*80)